# DeepTrace — Vision Transformer (ViT-Base) Fine-Tuning

This notebook fine-tunes the pretrained **`aaronkantrowitz/ai-image-detection`** model on domain-specific datasets (GenImage, FLUX, SDXL, modern diffusion) for the **DeepTrace SPHINX-26** project.

### Pipeline Overview:
1. **GPU Acceleration**: Verifies NVIDIA GPU availability.
2. **Base Architecture**: `google/vit-base-patch16-224-in21k` fine-tuned on CIFAKE (`aaronkantrowitz/ai-image-detection`).
3. **Label Semantics**: `0: REAL`, `1: FAKE` (strictly preserved).
4. **Transfer Learning**: Warmup + low learning rate (`1e-5`) with cosine/linear scheduling.
5. **Evaluation**: Comprehensive Accuracy, Precision, Recall, F1, and baseline comparison.

## Step 1: Environment & GPU Verification

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU not detected. Please enable GPU via Runtime -> Change runtime type -> T4/A100 GPU.")

## Step 2: Install Training Dependencies

In [ ]:
%pip install -q "transformers>=4.40.0" "datasets>=2.18.0" "accelerate>=0.28.0" "scikit-learn>=1.3.0" "pillow>=10.0.0" "safetensors>=0.4.0" "evaluate>=0.4.0"
print("Dependencies installed.")


## Step 3: Dataset Ingestion / Google Drive Mount

Mount your Google Drive to load the prepared `dataset/` folder, or upload a dataset `.zip` file.

In [ ]:
# Mount Google Drive (Recommended for persistent dataset storage)
from google.colab import drive
drive.mount('/content/drive')

# Set DATASET_DIR to your dataset location
# Expected structure:
# DATASET_DIR/
#   train/{real, fake}
#   validation/{real, fake}
#   test/{real, fake}
import os

DATASET_DIR = "/content/dataset"

# Example if unzipping from Google Drive:
# !unzip -q "/content/drive/MyDrive/DeepTrace/deeptrace_dataset.zip" -d /content/dataset

print(f"Dataset Path: {DATASET_DIR}")

## Step 4: Load Base Model & Image Processor

We load `aaronkantrowitz/ai-image-detection` and strictly retain its label mapping.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification, set_seed

BASE_MODEL_ID = "aaronkantrowitz/ai-image-detection"
set_seed(42)

print(f"Loading Image Processor from '{BASE_MODEL_ID}'...")
processor = AutoImageProcessor.from_pretrained(BASE_MODEL_ID)
print("Processor config:", processor)

print(f"Loading ViT Classification Model from '{BASE_MODEL_ID}'...")
model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL_ID,
    id2label={0: "REAL", 1: "FAKE"},
    label2id={"REAL": 0, "FAKE": 1},
    ignore_mismatched_sizes=False,
)
print("Model architectures:", model.config.architectures)
print("Model id2label:", model.config.id2label)

## Step 5: Dataset Preprocessing & Augmentation Transforms

In [ ]:
from datasets import load_dataset
from torchvision import transforms
import torch

# Load image dataset using Hugging Face datasets
dataset = load_dataset("imagefolder", data_dir=DATASET_DIR)
print(dataset)

# Define normalization values matching processor
image_mean = processor.image_mean if hasattr(processor, "image_mean") else [0.5, 0.5, 0.5]
image_std = processor.image_std if hasattr(processor, "image_std") else [0.5, 0.5, 0.5]
image_size = (processor.size["height"], processor.size["width"]) if isinstance(processor.size, dict) else (224, 224)

# Training transforms with data augmentation
train_transforms = transforms.Compose([
    transforms.Resize(image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_mean, std=image_std),
])

# Deterministic evaluation transforms
eval_transforms = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=image_mean, std=image_std),
])

def preprocess_train(examples):
    examples["pixel_values"] = [train_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples

def preprocess_eval(examples):
    examples["pixel_values"] = [eval_transforms(img.convert("RGB")) for img in examples["image"]]
    return examples

train_data = dataset["train"].with_transform(preprocess_train)
val_data = dataset["validation"].with_transform(preprocess_eval) if "validation" in dataset else None
test_data = dataset["test"].with_transform(preprocess_eval) if "test" in dataset else None

def collate_fn(batch):
    # ImageFolder: fake=0, real=1; intended: REAL=0, FAKE=1
    labels = torch.tensor([1 - x["label"] for x in batch])
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": labels,
    }

## Step 6: Metric Evaluation Functions

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    acc = accuracy_score(labels, predictions)
    prec = precision_score(labels, predictions, pos_label=1, zero_division=0)
    rec = recall_score(labels, predictions, pos_label=1, zero_division=0)
    f1 = f1_score(labels, predictions, pos_label=1, zero_division=0)
    f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)
    
    return {
        "accuracy": float(acc),
        "precision_fake": float(prec),
        "recall_fake": float(rec),
        "f1_fake": float(f1),
        "f1_macro": float(f1_macro),
    }

## Step 7: Configure Training Arguments & Trainer

We use transfer learning starting with a low learning rate (`1e-5`) and 5 epochs.

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_CHECKPOINT_DIR = "/content/checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_CHECKPOINT_DIR,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=25,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

## Step 8: Start Fine-Tuning

In [ ]:
# Execute training
train_result = trainer.train()
print("Training Complete!")
print(train_result.metrics)

## Step 9: Evaluate on Test Set & Compare with Baseline

In [ ]:
if test_data is not None:
    print("Evaluating fine-tuned model on test set...")
    finetuned_test_metrics = trainer.evaluate(test_data)
    print("Fine-Tuned Metrics:", finetuned_test_metrics)
    
    # Baseline comparison
    print("Evaluating original baseline model on test set for direct comparison...")
    baseline_model = AutoModelForImageClassification.from_pretrained(BASE_MODEL_ID)
    baseline_trainer = Trainer(
        model=baseline_model,
        compute_metrics=compute_metrics,
        data_collator=collate_fn,
    )
    baseline_metrics = baseline_trainer.evaluate(test_data)
    print("Baseline Metrics:", baseline_metrics)
    
    print("\n=== Comparative Test Performance ===")
    print(f"Baseline Accuracy:    {baseline_metrics.get('eval_accuracy', 0):.4f} | Fine-Tuned: {finetuned_test_metrics.get('eval_accuracy', 0):.4f}")
    print(f"Baseline F1-Score:    {baseline_metrics.get('eval_f1_macro', 0):.4f} | Fine-Tuned: {finetuned_test_metrics.get('eval_f1_macro', 0):.4f}")

## Step 10: Save Fine-Tuned Model Checkpoint

Save the final model in Hugging Face Transformers format (`model.safetensors`, `config.json`, `preprocessor_config.json`).

In [ ]:
FINAL_EXPORT_DIR = "/content/deeptrace-image-v1"
os.makedirs(FINAL_EXPORT_DIR, exist_ok=True)

# Save model weights & config
trainer.save_model(FINAL_EXPORT_DIR)
# Save processor config
processor.save_pretrained(FINAL_EXPORT_DIR)

print(f"Model and processor successfully saved to: {FINAL_EXPORT_DIR}")
print("Exported files:", os.listdir(FINAL_EXPORT_DIR))

## Step 11: Package Checkpoint for Download

Zip the checkpoint and download it to place in `backend/models/deeptrace-image-v1/`.

In [ ]:
!zip -r /content/deeptrace-image-v1.zip /content/deeptrace-image-v1

# Trigger browser download in Colab
from google.colab import files
files.download('/content/deeptrace-image-v1.zip')

# Or copy directly to Google Drive:
# !cp /content/deeptrace-image-v1.zip "/content/drive/MyDrive/DeepTrace/"